# WC 2026 Forecast Data

## Objective

Create the full 2026 forecast dataset. Start from the saved 2026 team-level stats, then match every team against every other team exactly once. Home and away are only labels for the model input columns, so each unordered matchup appears as one row.

## Inputs

- `1.DataCleaning-R/Data/CSV/WC2026Experience.csv`

## Outputs

- `1.DataCleaning-R/Data/CSV/WC2026ForecastData.csv`
- `1.DataCleaning-R/Data/RDS/WC2026ForecastData.rds`

## Packages

In [2]:
library(here)
library(tidyverse)

here() starts at /Users/eialnisman/Desktop/WC2026Forecast

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Team Stats

Load the saved 2026 team-level dataset and keep one row per team.

In [3]:
wc_2026_team_stats <- read_csv(
    here("1.DataCleaning-R", "Data", "CSV", "WC2026Experience.csv"),
    show_col_types = FALSE
) %>%
    rename(ELO = elo_2026) %>%
    mutate(
        team_id = coalesce(team_id, team_code),
        coach_name_nft = replace_na(coach_name_nft, "Unknown"),
        coach_full_name = replace_na(coach_full_name, "Unknown"),
        pre_wc_team_matches_coached = replace_na(pre_wc_team_matches_coached, 0)
    ) %>%
    distinct(team_name, .keep_all = TRUE) %>%
    arrange(team_name)

wc_2026_team_stats %>%
    summarize(
        teams = n(),
        duplicate_team_names = n() - n_distinct(team_name),
        .groups = "drop"
    )

teams,duplicate_team_names
<int>,<int>
48,0


## All Matchups

Use combinations instead of a full cross join so each matchup appears once. The team that sorts first alphabetically is labeled home.

In [4]:
team_index <- wc_2026_team_stats %>%
    mutate(team_index = row_number())

matchups <- combn(team_index$team_index, 2) %>%
    t() %>%
    as_tibble(.name_repair = "minimal") %>%
    set_names(c("home_index", "away_index"))

home_stats <- team_index %>%
    select(-team_index) %>%
    rename_with(~paste0("home_", .x))

away_stats <- team_index %>%
    select(-team_index) %>%
    rename_with(~paste0("away_", .x))

wc_2026_forecast_data <- matchups %>%
    mutate(match_id = str_c("WC2026_", str_pad(row_number(), 4, pad = "0")), .before = home_index) %>%
    left_join(
        team_index %>% select(home_index = team_index, home_team_name = team_name),
        by = "home_index"
    ) %>%
    left_join(
        team_index %>% select(away_index = team_index, away_team_name = team_name),
        by = "away_index"
    ) %>%
    left_join(home_stats, by = "home_team_name") %>%
    left_join(away_stats, by = "away_team_name") %>%
    select(-home_index, -away_index)

wc_2026_forecast_data %>%
    summarize(
        games = n(),
        unique_home_away_pairs = n_distinct(str_c(home_team_name, away_team_name, sep = "__")),
        repeated_unordered_pairs = n() - n_distinct(map2_chr(home_team_name, away_team_name, ~str_c(sort(c(.x, .y)), collapse = "__"))),
        .groups = "drop"
    )

games,unique_home_away_pairs,repeated_unordered_pairs
<int>,<int>,<int>
1128,1128,0


## Quick Checks

There should be 1,128 rows for 48 teams.

In [5]:
expected_games <- choose(nrow(wc_2026_team_stats), 2)

wc_2026_forecast_data %>%
    summarize(
        expected_games = expected_games,
        actual_games = n(),
        missing_values = sum(is.na(across(everything()))),
        .groups = "drop"
    )

wc_2026_forecast_data %>%
    select(match_id, home_team_name, away_team_name, home_ELO, away_ELO) %>%
    print(n = 10)

expected_games,actual_games,missing_values
<dbl>,<int>,<int>
1128,1128,0


# A tibble: 1,128 x 5
   match_id    home_team_name away_team_name         home_ELO away_ELO
   <chr>       <chr>          <chr>                     <dbl>    <dbl>
 1 WC2026_0001 Algeria        Argentina                  1757     2113
 2 WC2026_0002 Algeria        Australia                  1757     1774
 3 WC2026_0003 Algeria        Austria                    1757     1818
 4 WC2026_0004 Algeria        Belgium                    1757     1850
 5 WC2026_0005 Algeria        Bosnia and Herzegovina     1757     1572
 6 WC2026_0006 Algeria        Brazil                     1757     1978
 7 WC2026_0007 Algeria        Cabo Verde                 1757     1561
 8 WC2026_0008 Algeria        Canada                     1757     1802
 9 WC2026_0009 Algeria        Colombia                   1757     1998
10 WC2026_0010 Algeria        Congo DR                   1757     1657
# i 1,118 more rows


Inspect

In [6]:
wc_2026_forecast_data

match_id,home_team_name,away_team_name,home_tournament_id,home_team_id,home_team_code,home_distance_from_host_km,home_ELO,home_coach_name_nft,home_coach_full_name,...,away_ELO,away_coach_name_nft,away_coach_full_name,away_pre_wc_team_matches_coached,away_prior_world_cups_coached,away_squad_players,away_avg_age,away_missing_player_birth_dates,away_players_with_prior_wc,away_players_with_multiple_prior_wcs
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,...,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WC2026_0001,Algeria,Argentina,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,2113,"Scaloni, Lionel",Lionel Scaloni,94,1,26,29.05228,0,16,3
WC2026_0002,Algeria,Australia,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1774,"Popovi<U+0107>, Tony",Tony Popovi<U+0107>,16,0,26,27.35945,0,8,4
WC2026_0003,Algeria,Austria,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1818,"Rangnick, Ralf",Ralf Rangnick,44,0,25,28.60901,0,0,0
WC2026_0004,Algeria,Belgium,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1850,"Garcia, Rudi",Rudi Garcia,12,0,26,27.61449,0,11,6
WC2026_0005,Algeria,Bosnia and Herzegovina,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1572,"Barbarez, Sergej",Sergej Barbarez,20,0,26,26.42247,0,2,0
WC2026_0006,Algeria,Brazil,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1978,"Ancelotti, Carlo",Carlo Ancelotti,10,0,26,29.20276,0,5,0
WC2026_0007,Algeria,Cabo Verde,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1561,"Bubista,",Bubista,59,0,26,29.67641,0,0,0
WC2026_0008,Algeria,Canada,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1802,"Marsch, Jesse",Jesse Marsch,30,0,25,27.10330,0,10,0
WC2026_0009,Algeria,Colombia,WC-2026,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,...,1998,"Lorenzo, N<U+00E9>stor",N<U+00E9>stor Lorenzo,44,0,26,30.08930,0,9,5


## Save

Save the forecast-ready dataset.

In [7]:
write_csv(wc_2026_forecast_data, here("1.DataCleaning-R", "Data", "CSV", "WC2026ForecastData.csv"))
saveRDS(wc_2026_forecast_data, here("1.DataCleaning-R", "Data", "RDS", "WC2026ForecastData.rds"))